In [ ]:
import os

os.environ.setdefault("HF_HOME", "/nfsd/lttm4/tesisti/shahrampour/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_transformers")

for k in ["HF_HOME","HF_DATASETS_CACHE","TRANSFORMERS_CACHE"]:
    os.makedirs(os.environ[k], exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])

## 1) Imports

In [ ]:
import numpy as np
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
import copy
import torch.nn.functional as F

from peft import LoraConfig, get_peft_model, PeftModel
from safetensors.torch import load_file as safe_load_file
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
RUN_NAME = "cifar100_5step_simpleavg_normal_orth_replay_pretrained_n5"

RESULTS_DIR = os.path.join("results", RUN_NAME)
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

STEP1_OUT = f"outputs/{RUN_NAME}/step1_scratch"
STEP1_FINAL_OUT = f"outputs/{RUN_NAME}/step1_scratch_final"
JOINT_OUT = f"outputs/{RUN_NAME}/joint_full"
LORA_BANK_DIR = f"outputs/{RUN_NAME}/lora_bank"
MERGE_DIR = f"outputs/{RUN_NAME}/simple_avg_merge"

# New: RankExt bank, added without changing existing LoRA/merge paths
RANKEXT_BANK_DIR = f"outputs/{RUN_NAME}/rankext_bank"

os.makedirs(LORA_BANK_DIR, exist_ok=True)
os.makedirs(MERGE_DIR, exist_ok=True)
os.makedirs(RANKEXT_BANK_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(f"outputs/{RUN_NAME}", exist_ok=True)


# In this pretrained notebook, the "step1 final" path is actually the pretrained base.
# Keep both names to avoid changing the old pipeline cells.
PRETRAINED_BASE_OUT = STEP1_FINAL_OUT

## 2) Load CIFAR-100 (fine labels)

In [ ]:
from datasets import Image

dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")

dataset = dataset.cast_column("img", Image())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("example keys:", dataset["train"][0].keys())
print("first 10 classes:", class_names[:10])

In [ ]:
def make_custom_eval_dataset(class_ids, remap_labels=True):
    test_ds = filter_dataset_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        test_ds = test_ds.map(remap)
    else:
        label2new = None
        new2orig = None

    test_ds.set_transform(preprocess_val)
    return test_ds, label2new, new2orig

## 3) Define incremental class splits (2/5/10 steps)

In [ ]:

num_steps = 5  

assert num_classes % num_steps == 0
classes_per_step = num_classes // num_steps

class_splits = [
    list(range(s * classes_per_step, (s + 1) * classes_per_step))
    for s in range(num_steps)
]

print("classes per step:", classes_per_step)
print("split sizes:", [len(x) for x in class_splits])
print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])

In [ ]:
first_step_classes = class_splits[0]
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(class_splits[s])
all_classes = list(range(num_classes))

print("first step classes:", len(first_step_classes))
print("later step classes:", len(later_step_classes))
print("all classes:", len(all_classes))

## 4) Model + preprocessing

In [ ]:
# Requested model
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

from torchvision import transforms
from PIL import Image
import numpy as np
import torch

size = image_processor.size
if isinstance(size, dict):
    H = size.get("height", 224)
    W = size.get("width", 224)
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = x.astype(np.uint8)
        arr = np.squeeze(arr)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if not (arr.ndim == 3 and arr.shape[-1] in (1, 3)):
            raise TypeError(f"Unexpected image array shape after squeeze: {arr.shape}")

        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean().item()}

## 5) Build per-step datasets (step / cumulative / full)

In [ ]:
def classes_for_step(step_idx: int) -> list[int]:
    return class_splits[step_idx]

def classes_for_cumulative(step_idx: int) -> list[int]:
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def filter_by_classes(ds, class_ids):
    class_set = set(class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda x: int(x["label"]) in class_set)

def make_step_datasets(step_idx: int, split_type: str = "new_only", remap_labels: bool = False):
    """
    split_type:
      - 'new_only'   : only classes of this step
      - 'cumulative' : union of classes up to this step
      - 'full'       : all classes (100)
    """
    if split_type == "full":
        class_ids = list(range(num_classes))
    elif split_type == "cumulative":
        class_ids = classes_for_cumulative(step_idx)
    elif split_type == "new_only":
        class_ids = classes_for_step(step_idx)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")

    train_ds = filter_by_classes(dataset["train"], class_ids)
    test_ds  = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        train_ds = train_ds.map(remap)
        test_ds  = test_ds.map(remap)
    else:
        label2new = {c: c for c in class_ids}
        new2orig = {c: c for c in class_ids}

    train_ds = train_ds.with_transform(preprocess_train)
    test_ds = test_ds.with_transform(preprocess_val)

    print(f"[Dataset] Step {step_idx+1} | split_type={split_type}")
    print(f"[Dataset] Classes: {class_ids[:5]} ... {class_ids[-5:]}")
    print(f"[Dataset] Train size: {len(train_ds)} | Test size: {len(test_ds)}")

    return train_ds, test_ds, label2new, new2orig, class_ids

eval_all = dataset["test"].with_transform(preprocess_val)

print("eval_all size:", len(eval_all))

## 6) Training recipes (reasonable settings)

In [ ]:
set_seed(42)

# =========================
# DEBUG / FAST SETTINGS
# =========================
# Keep same variable names to avoid breaking later cells.

DEBUG_MODE = True

SCRATCH_EPOCHS = 1
LORA_EPOCHS    = 1
FT_EPOCHS      = 1
JOINT_EPOCHS   = 1

BATCH_SCRATCH = 8
ACCUM_SCRATCH = 2

BATCH_LORA    = 16
ACCUM_LORA    = 1

BATCH_FT      = 8
ACCUM_FT      = 2

LR_SCRATCH = 5e-5
LR_LORA    = 5e-5
LR_FT      = 3e-5
LR_JOINT   = 5e-5

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# =========================
# Full FT + Replay diagnostic baseline
# =========================
FT_REPLAY_EPOCHS = FT_EPOCHS
FT_REPLAY_PER_OLD_STEP = 250
LR_FT_REPLAY = LR_FT

# =========================
# Improved Orthogonality settings
# =========================
# We replace the old weak orth penalty with normalized delta-space cosine loss.
# Keep name LAMBDA_ORTH through run_config in Cell 16.
ORTH_LOSS_MODE = "delta_cosine_squared"
ORTH_USE_PREVIOUS_AVERAGE = False
ORTH_BASE_WEIGHT_COEF = 0.0

# Debug lambdas. First use 0.10. Later we can sweep [0.03, 0.10, 0.30, 1.00].
LAMBDA_ORTH_VALUE = 0.10

# =========================
# RankExt debugging settings
# =========================
# First debug plain true RankExt only.
# Do NOT use IPC-lite yet because current results suggest it may block new-task learning.
RANKEXT_NEW_RANK_PER_STEP = 8
RANKEXT_ALPHA_MULTIPLIER = 2
RANKEXT_EPOCHS = LORA_EPOCHS

LR_RANKEXT = 5e-5

FREEZE_OLD_LORA_RANKS = True
FREEZE_OLD_CLASSIFIER_ROWS = True

# Temporarily disabled for debugging.
USE_RANKEXT_IPC_LITE = False
RANKEXT_IPC_TOP_FRACTION = 0.10

# Keep weight decay zero because old slices are frozen through gradient masking.
RANKEXT_WEIGHT_DECAY = 0.0

# First isolate pure RankExt. After it learns current steps, turn this back on.
LAMBDA_RANKEXT_ORTH_VALUE = 0.0

results = []

print("=" * 80)
print("DEBUG_MODE:", DEBUG_MODE)
print("SCRATCH_EPOCHS:", SCRATCH_EPOCHS)
print("LORA_EPOCHS:", LORA_EPOCHS)
print("FT_EPOCHS:", FT_EPOCHS)
print("JOINT_EPOCHS:", JOINT_EPOCHS)
print("FT_REPLAY_EPOCHS:", FT_REPLAY_EPOCHS)
print("FT_REPLAY_PER_OLD_STEP:", FT_REPLAY_PER_OLD_STEP)
print("LAMBDA_ORTH_VALUE:", LAMBDA_ORTH_VALUE)
print("ORTH_LOSS_MODE:", ORTH_LOSS_MODE)
print("RANKEXT_EPOCHS:", RANKEXT_EPOCHS)
print("LR_RANKEXT:", LR_RANKEXT)
print("USE_RANKEXT_IPC_LITE:", USE_RANKEXT_IPC_LITE)
print("LAMBDA_RANKEXT_ORTH_VALUE:", LAMBDA_RANKEXT_ORTH_VALUE)
print("=" * 80)

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "model_checkpoint": model_checkpoint,
    "init_mode": "pretrained_imagenet",
    "num_classes": num_classes,
    "num_steps": num_steps,
    "classes_per_step": classes_per_step,

    "scratch_epochs": SCRATCH_EPOCHS,
    "lora_epochs": LORA_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "joint_epochs": JOINT_EPOCHS,

    "lr_scratch": LR_SCRATCH,
    "lr_lora": LR_LORA,
    "lr_ft": LR_FT,
    "lr_joint": LR_JOINT,

    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.1,
    "target_modules": ["query", "key", "value"],

    "variants": ["normal", "orth", "replay"],

    # Improved orthogonality
    "lambda_orth": LAMBDA_ORTH_VALUE,
    "orth_loss_mode": ORTH_LOSS_MODE,
    "orth_use_previous_average": ORTH_USE_PREVIOUS_AVERAGE,
    "orth_base_weight_coef": ORTH_BASE_WEIGHT_COEF,

    "replay_per_old_step": 250,
    "base_source": "imagenet_pretrained_vit",

    # DO-Merge
    "use_do_merge": True,
    "do_merge_source_variant": "normal",

    # FT + Replay diagnostic
    "ft_replay_epochs": FT_REPLAY_EPOCHS,
    "ft_replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
    "lr_ft_replay": LR_FT_REPLAY,

    # RankExt config
    "rankext_new_rank_per_step": RANKEXT_NEW_RANK_PER_STEP,
    "rankext_alpha_multiplier": RANKEXT_ALPHA_MULTIPLIER,
    "rankext_epochs": RANKEXT_EPOCHS,
    "lr_rankext": LR_RANKEXT,
    "rankext_weight_decay": RANKEXT_WEIGHT_DECAY,
    "freeze_old_lora_ranks": FREEZE_OLD_LORA_RANKS,
    "freeze_old_classifier_rows": FREEZE_OLD_CLASSIFIER_ROWS,
    "use_rankext_ipc_lite": USE_RANKEXT_IPC_LITE,
    "rankext_ipc_top_fraction": RANKEXT_IPC_TOP_FRACTION,
    "lambda_rankext_orth": LAMBDA_RANKEXT_ORTH_VALUE,
}

with open(os.path.join(METRICS_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

LORA_R = run_config["lora_r"]
LORA_ALPHA = run_config["lora_alpha"]
LORA_DROPOUT = run_config["lora_dropout"]
TARGET_MODULES = run_config["target_modules"]

VARIANTS = run_config["variants"]

LAMBDA_ORTH = run_config["lambda_orth"]
ORTH_LOSS_MODE = run_config["orth_loss_mode"]
ORTH_USE_PREVIOUS_AVERAGE = run_config["orth_use_previous_average"]
ORTH_BASE_WEIGHT_COEF = run_config["orth_base_weight_coef"]

REPLAY_PER_OLD_STEP = run_config["replay_per_old_step"]

USE_DO_MERGE = run_config["use_do_merge"]
DO_MERGE_SOURCE_VARIANT = run_config["do_merge_source_variant"]

FT_REPLAY_EPOCHS = run_config["ft_replay_epochs"]
FT_REPLAY_PER_OLD_STEP = run_config["ft_replay_per_old_step"]
LR_FT_REPLAY = run_config["lr_ft_replay"]

RANKEXT_NEW_RANK_PER_STEP = run_config["rankext_new_rank_per_step"]
RANKEXT_ALPHA_MULTIPLIER = run_config["rankext_alpha_multiplier"]
RANKEXT_EPOCHS = run_config["rankext_epochs"]
LR_RANKEXT = run_config["lr_rankext"]
RANKEXT_WEIGHT_DECAY = run_config["rankext_weight_decay"]
FREEZE_OLD_LORA_RANKS = run_config["freeze_old_lora_ranks"]
FREEZE_OLD_CLASSIFIER_ROWS = run_config["freeze_old_classifier_rows"]
USE_RANKEXT_IPC_LITE = run_config["use_rankext_ipc_lite"]
RANKEXT_IPC_TOP_FRACTION = run_config["rankext_ipc_top_fraction"]
LAMBDA_RANKEXT_ORTH = run_config["lambda_rankext_orth"]

print(json.dumps(run_config, indent=2))

## 7) Pretrained ViT base with CIFAR-100 classifier

In [ ]:
import os
import shutil
import json

if os.path.exists(PRETRAINED_BASE_OUT):
    shutil.rmtree(PRETRAINED_BASE_OUT)

os.makedirs(PRETRAINED_BASE_OUT, exist_ok=True)

config_base = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

base_model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    config=config_base,
    ignore_mismatched_sizes=True,
)

print("Base init mode: pretrained ImageNet ViT")
print("Base model num_labels:", base_model.config.num_labels)
print("Base classifier out_features:", base_model.classifier.out_features)

assert base_model.config.num_labels == num_classes
assert base_model.classifier.out_features == num_classes

base_model.save_pretrained(PRETRAINED_BASE_OUT, safe_serialization=False)
image_processor.save_pretrained(PRETRAINED_BASE_OUT)

print("Saved pretrained base model to:", PRETRAINED_BASE_OUT)
print("Files in PRETRAINED_BASE_OUT:", os.listdir(PRETRAINED_BASE_OUT))

reloaded_base = AutoModelForImageClassification.from_pretrained(PRETRAINED_BASE_OUT)
print("Reloaded base num_labels:", reloaded_base.config.num_labels)
print("Reloaded base classifier out_features:", reloaded_base.classifier.out_features)

assert reloaded_base.config.num_labels == num_classes
assert reloaded_base.classifier.out_features == num_classes

results = []

In [ ]:
print("Init mode:", run_config["init_mode"])
assert run_config["init_mode"] == "pretrained_imagenet"
assert os.path.exists(PRETRAINED_BASE_OUT)

## 8) Step 2: LoRA only (freeze backbone) on top of Step 1 model

In [ ]:
first_step_classes = classes_for_step(0)

def make_eval_dataset(class_ids, name=None):
    test_ds = filter_by_classes(dataset["test"], class_ids)
    test_ds = test_ds.with_transform(preprocess_val)
    if name is not None:
        print(f"[Eval set] {name}: num_classes={len(class_ids)}, size={len(test_ds)}")
    return test_ds

In [ ]:
def normalize_module_name(module_name):
    prefixes = [
        "base_model.model.",
        "base_model.",
        "model.",
    ]
    for p in prefixes:
        if module_name.startswith(p):
            module_name = module_name[len(p):]
    module_name = module_name.replace("vit.model.", "vit.")
    return module_name


def lambda_tag(x):
    return str(x).replace(".", "p").replace("-", "m")


def load_adapter_state(adapter_dir):
    safe_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    bin_path = os.path.join(adapter_dir, "adapter_model.bin")

    if os.path.exists(safe_path):
        return safe_load_file(safe_path)

    if os.path.exists(bin_path):
        return torch.load(bin_path, map_location="cpu")

    raise FileNotFoundError(f"No adapter weights found in {adapter_dir}")


def get_lora_scaling_factor(adapter_dir):
    from peft import PeftConfig
    peft_cfg = PeftConfig.from_pretrained(adapter_dir)
    return peft_cfg.lora_alpha / peft_cfg.r


def extract_lora_factor_matrices(adapter_state):
    lora_factor_matrices = {}

    for param_name, param_tensor in adapter_state.items():
        if ".lora_A." in param_name:
            base_module_name = normalize_module_name(param_name.split(".lora_A.")[0])
            lora_factor_matrices.setdefault(base_module_name, {})["A_matrix"] = (
                param_tensor.detach().cpu().float()
            )

        elif ".lora_B." in param_name:
            base_module_name = normalize_module_name(param_name.split(".lora_B.")[0])
            lora_factor_matrices.setdefault(base_module_name, {})["B_matrix"] = (
                param_tensor.detach().cpu().float()
            )

    return {
        module_name: factors
        for module_name, factors in lora_factor_matrices.items()
        if "A_matrix" in factors and "B_matrix" in factors
    }


def sample_replay_from_previous_steps(step_idx, per_old_step=250, seed=42):
    replay_parts = []

    for prev_step_idx in range(step_idx):
        prev_train, _, _, _, _ = make_step_datasets(
            prev_step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        n_take = min(per_old_step, len(prev_train))
        replay_part = prev_train.shuffle(seed=seed + prev_step_idx).select(range(n_take))
        replay_parts.append(replay_part)

    if len(replay_parts) == 0:
        return None

    return concatenate_datasets(replay_parts)


def build_variant_train_dataset(step_idx, variant_name):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    if variant_name == "replay":
        replay_ds = sample_replay_from_previous_steps(
            step_idx=step_idx,
            per_old_step=REPLAY_PER_OLD_STEP,
            seed=42,
        )

        if replay_ds is not None:
            train_step = concatenate_datasets([train_step, replay_ds]).shuffle(
                seed=100 + step_idx
            )

    return train_step, test_step, label2new, new2orig, class_ids


def extract_delta_bank_from_adapter(adapter_dir):
    adapter_state = load_adapter_state(adapter_dir)
    scaling = get_lora_scaling_factor(adapter_dir)

    module_to_factor_matrices = extract_lora_factor_matrices(adapter_state)
    delta_bank = {}

    for module_name, factor_dict in module_to_factor_matrices.items():
        A_i = factor_dict["A_matrix"]
        B_i = factor_dict["B_matrix"]
        delta_bank[module_name] = (B_i @ A_i) * scaling

    return delta_bank


def build_prev_delta_banks(adapter_dirs):
    banks = []

    for adapter_dir in adapter_dirs:
        banks.append(extract_delta_bank_from_adapter(adapter_dir))

    return banks


def build_average_prev_delta_bank(prev_delta_banks):
    if prev_delta_banks is None or len(prev_delta_banks) == 0:
        return {}

    module_names = sorted(
        set().union(*[set(bank.keys()) for bank in prev_delta_banks])
    )

    avg_bank = {}

    for module_name in module_names:
        tensors = []

        for bank in prev_delta_banks:
            if module_name in bank:
                tensors.append(bank[module_name].float())

        if len(tensors) > 0:
            avg_bank[module_name] = torch.stack(tensors, dim=0).mean(dim=0)

    return avg_bank


def compute_orth_penalty_from_model(
    model,
    prev_delta_banks,
    base_weight_coef=0.0,
):
    """
    Improved orthogonality loss.

    Main loss:
        normalized squared cosine between current LoRA delta and previous LoRA deltas.

    This is more stable than raw trace because it normalizes layer scale.
    """
    penalties = []

    if prev_delta_banks is None:
        prev_delta_banks = []

    if ORTH_USE_PREVIOUS_AVERAGE and len(prev_delta_banks) > 0:
        comparison_banks = [build_average_prev_delta_bank(prev_delta_banks)]
    else:
        comparison_banks = prev_delta_banks

    device_now = next(model.parameters()).device

    for module_name, module in model.named_modules():
        module_name_norm = normalize_module_name(module_name)

        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        if hasattr(module, "scaling"):
            scaling = (
                module.scaling["default"]
                if isinstance(module.scaling, dict)
                else module.scaling
            )
        else:
            scaling = 1.0

        curr_delta = (B @ A) * float(scaling)
        curr_flat = curr_delta.reshape(-1)

        curr_norm = curr_flat.norm()

        if curr_norm.detach().item() < 1e-12:
            continue

        for prev_bank in comparison_banks:
            if module_name_norm not in prev_bank:
                continue

            prev_delta = prev_bank[module_name_norm].to(
                device=curr_delta.device,
                dtype=curr_delta.dtype,
            )

            prev_flat = prev_delta.reshape(-1)
            prev_norm = prev_flat.norm()

            if prev_norm.detach().item() < 1e-12:
                continue

            denom = curr_norm * prev_norm + 1e-8
            cos_overlap = torch.dot(curr_flat, prev_flat) / denom

            if ORTH_LOSS_MODE == "delta_cosine_abs":
                penalties.append(torch.abs(cos_overlap))
            else:
                penalties.append(cos_overlap.pow(2))

        if base_weight_coef > 0 and hasattr(module, "base_layer"):
            base_w = module.base_layer.weight

            if base_w.shape == curr_delta.shape:
                base_flat = base_w.detach().reshape(-1).to(
                    device=curr_delta.device,
                    dtype=curr_delta.dtype,
                )

                base_norm = base_flat.norm()

                if base_norm.detach().item() > 1e-12:
                    denom = curr_norm * base_norm + 1e-8
                    base_cos_overlap = torch.dot(curr_flat, base_flat) / denom
                    penalties.append(base_weight_coef * base_cos_overlap.pow(2))

    if len(penalties) == 0:
        return torch.tensor(0.0, device=device_now)

    return torch.stack(penalties).mean()


class OrthogonalLoRATrainer(Trainer):
    def __init__(
        self,
        *args,
        prev_delta_banks=None,
        lambda_orth=0.03,
        orth_base_weight_coef=0.0,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.prev_delta_banks = prev_delta_banks if prev_delta_banks is not None else []
        self.lambda_orth = lambda_orth
        self.orth_base_weight_coef = orth_base_weight_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        base_loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss

        if len(self.prev_delta_banks) > 0 and self.lambda_orth > 0:
            orth_penalty = compute_orth_penalty_from_model(
                model=model,
                prev_delta_banks=self.prev_delta_banks,
                base_weight_coef=self.orth_base_weight_coef,
            )
            loss = base_loss + self.lambda_orth * orth_penalty
        else:
            orth_penalty = torch.tensor(0.0, device=base_loss.device)
            loss = base_loss

        return (loss, outputs) if return_outputs else loss

In [ ]:
def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)
    submodule = model

    for part in module_name.split("."):
        submodule = getattr(submodule, part)

    return submodule


def extract_classifier_tensors(adapter_state):
    classifier_weight_key = None
    classifier_bias_key = None

    for param_name in adapter_state.keys():
        if "classifier" in param_name and param_name.endswith("weight"):
            classifier_weight_key = param_name

        if "classifier" in param_name and param_name.endswith("bias"):
            classifier_bias_key = param_name

    if classifier_weight_key is None:
        return None, None

    classifier_weight_tensor = adapter_state[classifier_weight_key].detach().cpu().float()

    classifier_bias_tensor = (
        adapter_state[classifier_bias_key].detach().cpu().float()
        if classifier_bias_key is not None
        else None
    )

    return classifier_weight_tensor, classifier_bias_tensor


def stitch_classifier_rows(base_model_W0, classifier_row_sources):
    with torch.no_grad():
        for class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i in classifier_row_sources:
            if classifier_weight_i is None:
                continue

            base_model_W0.classifier.weight.data[class_ids_for_this_adapter] = (
                classifier_weight_i[class_ids_for_this_adapter].to(
                    base_model_W0.classifier.weight.device,
                    dtype=base_model_W0.classifier.weight.dtype,
                )
            )

            if classifier_bias_i is not None and base_model_W0.classifier.bias is not None:
                base_model_W0.classifier.bias.data[class_ids_for_this_adapter] = (
                    classifier_bias_i[class_ids_for_this_adapter].to(
                        base_model_W0.classifier.bias.device,
                        dtype=base_model_W0.classifier.bias.dtype,
                    )
                )


def build_fixed_avg_merged_model(base_model_dir, adapter_dirs, adapter_class_groups):
    base_model_W0 = AutoModelForImageClassification.from_pretrained(base_model_dir)
    base_model_W0.eval()

    module_to_delta_list = {}
    classifier_row_sources = []

    for adapter_dir, class_ids_for_this_adapter in zip(adapter_dirs, adapter_class_groups):
        adapter_state = load_adapter_state(adapter_dir)
        lora_scaling_factor_si = get_lora_scaling_factor(adapter_dir)
        module_to_factor_matrices = extract_lora_factor_matrices(adapter_state)

        for module_name, factor_dict in module_to_factor_matrices.items():
            A_i = factor_dict["A_matrix"]
            B_i = factor_dict["B_matrix"]

            effective_lora_delta = (B_i @ A_i) * lora_scaling_factor_si
            module_to_delta_list.setdefault(module_name, []).append(effective_lora_delta)

        classifier_weight_i, classifier_bias_i = extract_classifier_tensors(adapter_state)
        classifier_row_sources.append(
            (class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i)
        )

    with torch.no_grad():
        for module_name, delta_list in module_to_delta_list.items():
            avg_delta = torch.stack(delta_list, dim=0).mean(dim=0)
            target_module = get_submodule_by_name(base_model_W0, module_name)

            if target_module.weight.shape != avg_delta.shape:
                raise ValueError(
                    f"Shape mismatch for {module_name}: "
                    f"weight={target_module.weight.shape}, delta={avg_delta.shape}"
                )

            target_module.weight.data += avg_delta.to(
                target_module.weight.device,
                dtype=target_module.weight.dtype,
            )

    stitch_classifier_rows(base_model_W0, classifier_row_sources)

    base_model_W0.eval()
    return base_model_W0


def orthogonalize_direction_columns(direction_list):
    """
    Data-free lightweight orthogonalization.

    direction_list:
        list of tensors with shape [out_features, in_features]

    For each input column, we orthogonalize the task direction vectors
    across adapters using Gram-Schmidt, then average them.
    """
    if len(direction_list) == 1:
        return direction_list[0]

    stacked_dirs = torch.stack(direction_list, dim=0)
    num_tasks, out_dim, in_dim = stacked_dirs.shape

    merged_direction = torch.zeros_like(stacked_dirs[0])

    for col_idx in range(in_dim):
        basis = []
        col_vectors = [stacked_dirs[t, :, col_idx].clone() for t in range(num_tasks)]

        ortho_vectors = []

        for v in col_vectors:
            v_work = v.clone()

            for b in basis:
                v_work = v_work - torch.dot(v_work, b) * b

            norm_v = v_work.norm()

            if norm_v > 1e-8:
                q = v_work / norm_v
                basis.append(q)
                ortho_vectors.append(q)
            else:
                original_norm = v.norm()
                if original_norm > 1e-8:
                    ortho_vectors.append(v / original_norm)

        if len(ortho_vectors) == 0:
            continue

        avg_col = torch.stack(ortho_vectors, dim=0).mean(dim=0)
        avg_col_norm = avg_col.norm()

        if avg_col_norm > 1e-8:
            avg_col = avg_col / avg_col_norm

        merged_direction[:, col_idx] = avg_col

    return merged_direction


def decoupled_orthogonal_merge_delta(delta_list):
    """
    Paper-inspired DO-Merge lite.

    For each LoRA delta:
        delta = direction * magnitude

    Magnitude:
        column L2 norm

    Direction:
        column-normalized delta

    Merge:
        average magnitudes
        orthogonalize + average directions
        reconstruct delta
    """
    eps = 1e-8

    magnitude_list = []
    direction_list = []

    for delta in delta_list:
        delta = delta.detach().cpu().float()
        magnitude = delta.norm(dim=0, keepdim=True).clamp_min(eps)
        direction = delta / magnitude

        magnitude_list.append(magnitude)
        direction_list.append(direction)

    merged_magnitude = torch.stack(magnitude_list, dim=0).mean(dim=0)
    merged_direction = orthogonalize_direction_columns(direction_list)

    merged_delta = merged_direction * merged_magnitude

    return merged_delta


def build_do_merged_model(base_model_dir, adapter_dirs, adapter_class_groups):
    """
    DO-Merge lite:
    - Uses LoRA delta = B @ A in full-rank space
    - Decouples magnitude and direction
    - Orthogonalizes directions data-free
    - Stitches classifier rows like Simple AVG
    """
    base_model_W0 = AutoModelForImageClassification.from_pretrained(base_model_dir)
    base_model_W0.eval()

    module_to_delta_list = {}
    classifier_row_sources = []

    for adapter_dir, class_ids_for_this_adapter in zip(adapter_dirs, adapter_class_groups):
        adapter_state = load_adapter_state(adapter_dir)
        lora_scaling_factor_si = get_lora_scaling_factor(adapter_dir)
        module_to_factor_matrices = extract_lora_factor_matrices(adapter_state)

        for module_name, factor_dict in module_to_factor_matrices.items():
            A_i = factor_dict["A_matrix"]
            B_i = factor_dict["B_matrix"]

            effective_lora_delta = (B_i @ A_i) * lora_scaling_factor_si
            module_to_delta_list.setdefault(module_name, []).append(effective_lora_delta)

        classifier_weight_i, classifier_bias_i = extract_classifier_tensors(adapter_state)
        classifier_row_sources.append(
            (class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i)
        )

    with torch.no_grad():
        for module_name, delta_list in module_to_delta_list.items():
            merged_delta = decoupled_orthogonal_merge_delta(delta_list)
            target_module = get_submodule_by_name(base_model_W0, module_name)

            if target_module.weight.shape != merged_delta.shape:
                raise ValueError(
                    f"Shape mismatch for {module_name}: "
                    f"weight={target_module.weight.shape}, delta={merged_delta.shape}"
                )

            target_module.weight.data += merged_delta.to(
                target_module.weight.device,
                dtype=target_module.weight.dtype,
            )

    stitch_classifier_rows(base_model_W0, classifier_row_sources)

    base_model_W0.eval()
    return base_model_W0

In [ ]:
lora_results = []

first_step_classes = classes_for_step(0)


def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)


for variant_name in VARIANTS:
    print("\n" + "=" * 80)
    print(f"LoRA variant: {variant_name.upper()}")
    print("=" * 80)

    variant_bank_dir = os.path.join(LORA_BANK_DIR, variant_name)

    if os.path.exists(variant_bank_dir):
        shutil.rmtree(variant_bank_dir)

    os.makedirs(variant_bank_dir, exist_ok=True)

    base_model_dir = STEP1_FINAL_OUT

    for step_idx in range(num_steps):
        train_step, test_step, label2new, new2orig, class_ids = build_variant_train_dataset(
            step_idx,
            variant_name,
        )

        print(f"\n[{variant_name}] Step {step_idx + 1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])
        print("Train size:", len(train_step), "| Test size:", len(test_step))

        tr_min, tr_max = _label_range(train_step)
        te_min, te_max = _label_range(test_step)

        print("Train label range:", tr_min, tr_max)
        print("Test label range :", te_min, te_max)

        base_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            target_modules=TARGET_MODULES,
            modules_to_save=["classifier"],
        )

        model_lora = get_peft_model(base_model, lora_config)

        step_train_dir = f"outputs/{RUN_NAME}/step_{step_idx + 1}_lora_{variant_name}"

        args_lora = TrainingArguments(
            output_dir=step_train_dir,
            remove_unused_columns=False,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            num_train_epochs=LORA_EPOCHS,
            learning_rate=LR_LORA,
            weight_decay=0.01,
            warmup_ratio=WARMUP_RATIO,
            lr_scheduler_type=SCHED,
            per_device_train_batch_size=BATCH_LORA,
            per_device_eval_batch_size=32,
            gradient_accumulation_steps=ACCUM_LORA,
            fp16=USE_FP16,
            dataloader_num_workers=4,
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            greater_is_better=True,
            report_to="none",
            max_grad_norm=1.0,
        )

        if variant_name == "orth" and step_idx > 0:
            prev_adapter_dirs = [
                os.path.join(variant_bank_dir, f"step_{s}_adapter")
                for s in range(1, step_idx + 1)
            ]

            prev_delta_banks = build_prev_delta_banks(prev_adapter_dirs)

            print(f"[Orth train-loss] previous banks: {len(prev_delta_banks)}")
            print(f"[Orth train-loss] lambda_orth: {LAMBDA_ORTH}")
            print(f"[Orth train-loss] base_weight_coef: {ORTH_BASE_WEIGHT_COEF}")

            trainer_lora = OrthogonalLoRATrainer(
                model=model_lora,
                args=args_lora,
                train_dataset=train_step,
                eval_dataset=test_step,
                data_collator=collate_fn,
                compute_metrics=compute_metrics,
                prev_delta_banks=prev_delta_banks,
                lambda_orth=LAMBDA_ORTH,
                orth_base_weight_coef=ORTH_BASE_WEIGHT_COEF,
            )
        else:
            trainer_lora = Trainer(
                model=model_lora,
                args=args_lora,
                train_dataset=train_step,
                eval_dataset=test_step,
                data_collator=collate_fn,
                compute_metrics=compute_metrics,
            )

        trainer_lora.train()
        eval_current = trainer_lora.evaluate(eval_dataset=test_step)

        eval_first = make_eval_dataset(classes_for_step(0))
        seen_now = classes_for_cumulative(step_idx)
        later_now = [c for c in seen_now if c not in classes_for_step(0)]

        eval_later = make_eval_dataset(later_now) if len(later_now) > 0 else None
        eval_seen = make_eval_dataset(seen_now)

        eval_first_res = trainer_lora.evaluate(eval_dataset=eval_first)
        eval_later_res = (
            trainer_lora.evaluate(eval_dataset=eval_later)
            if eval_later is not None
            else {"eval_accuracy": np.nan, "eval_loss": np.nan}
        )
        eval_seen_res = trainer_lora.evaluate(eval_dataset=eval_seen)

        pd.DataFrame(trainer_lora.state.log_history).to_csv(
            os.path.join(
                TABLES_DIR,
                f"step{step_idx + 1}_lora_{variant_name}_log_history.csv",
            ),
            index=False,
        )

        step_adapter_dir = os.path.join(
            variant_bank_dir,
            f"step_{step_idx + 1}_adapter",
        )
        os.makedirs(step_adapter_dir, exist_ok=True)

        trainer_lora.model.save_pretrained(step_adapter_dir)
        image_processor.save_pretrained(step_adapter_dir)

        adapter_meta = {
            "step": step_idx + 1,
            "variant": variant_name,
            "base_model_dir": STEP1_FINAL_OUT,
            "class_ids": class_ids,
            "num_labels": num_classes,
        }

        with open(os.path.join(step_adapter_dir, "adapter_meta.json"), "w") as f:
            json.dump(adapter_meta, f, indent=2)

        lora_results.extend([
            {
                "experiment": f"{variant_name}_lora_step_{step_idx + 1}",
                "method": f"lora_{variant_name}",
                "step": step_idx + 1,
                "eval_type": "current_step",
                "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_current.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"{variant_name}_lora_step_{step_idx + 1}",
                "method": f"lora_{variant_name}",
                "step": step_idx + 1,
                "eval_type": "first_step",
                "eval_accuracy": float(eval_first_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_first_res.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"{variant_name}_lora_step_{step_idx + 1}",
                "method": f"lora_{variant_name}",
                "step": step_idx + 1,
                "eval_type": "later_steps",
                "eval_accuracy": float(eval_later_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_later_res.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"{variant_name}_lora_step_{step_idx + 1}",
                "method": f"lora_{variant_name}",
                "step": step_idx + 1,
                "eval_type": "all_seen",
                "eval_accuracy": float(eval_seen_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_seen_res.get("eval_loss", np.nan)),
            },
        ])

print("\nAll LoRA variants finished.")

In [ ]:
rankext_results = []
rankext_classifier_snapshots = {}

rankext_debug_rows = []


def find_state_key(param_name, state_dict):
    candidates = [
        param_name,
        param_name.replace(".default.", "."),
        param_name.replace(".modules_to_save.default", ""),
        param_name.replace(".original_module", ""),
    ]

    for k in candidates:
        if k in state_dict:
            return k

    short = param_name.replace(".default.", ".")
    short = short.split("base_model.model.")[-1]

    for k in state_dict.keys():
        if k.endswith(short):
            return k

    return None


def copy_previous_rankext_weights(model_lora, prev_adapter_dir, r_old):
    if prev_adapter_dir is None or r_old == 0:
        print("[RankExt] No previous adapter to copy.")
        return {
            "copied_A": 0,
            "copied_B": 0,
            "copied_classifier": 0,
        }

    prev_sd = load_adapter_state(prev_adapter_dir)

    copied_A = 0
    copied_B = 0
    copied_classifier = 0

    with torch.no_grad():
        for name, p in model_lora.named_parameters():
            key = find_state_key(name, prev_sd)

            if key is None:
                continue

            old_p = prev_sd[key].to(device=p.device, dtype=p.dtype)

            if "lora_A" in name and "weight" in name:
                rows = min(r_old, old_p.shape[0], p.shape[0])
                p[:rows, :].copy_(old_p[:rows, :])
                copied_A += 1

            elif "lora_B" in name and "weight" in name:
                cols = min(r_old, old_p.shape[1], p.shape[1])
                p[:, :cols].copy_(old_p[:, :cols])
                copied_B += 1

            elif "classifier" in name and p.shape == old_p.shape:
                p.copy_(old_p)
                copied_classifier += 1

    print(
        f"[RankExt] copied A={copied_A}, "
        f"B={copied_B}, classifier={copied_classifier}"
    )

    return {
        "copied_A": copied_A,
        "copied_B": copied_B,
        "copied_classifier": copied_classifier,
    }


def freeze_old_lora_ranks(model_lora, r_old):
    if not FREEZE_OLD_LORA_RANKS or r_old == 0:
        print("[RankExt] No old LoRA ranks to freeze.")
        return {
            "frozen_A_modules": 0,
            "frozen_B_modules": 0,
        }

    def hook_A(grad):
        grad = grad.clone()
        grad[:r_old, :] = 0.0
        return grad

    def hook_B(grad):
        grad = grad.clone()
        grad[:, :r_old] = 0.0
        return grad

    nA, nB = 0, 0

    for _, module in model_lora.named_modules():
        if hasattr(module, "lora_A") and "default" in module.lora_A:
            module.lora_A["default"].weight.register_hook(hook_A)
            nA += 1

        if hasattr(module, "lora_B") and "default" in module.lora_B:
            module.lora_B["default"].weight.register_hook(hook_B)
            nB += 1

    print(f"[RankExt] frozen old ranks: A modules={nA}, B modules={nB}")

    return {
        "frozen_A_modules": nA,
        "frozen_B_modules": nB,
    }


def get_trainable_classifier(model_lora):
    cw = model_lora.base_model.model.classifier

    if hasattr(cw, "modules_to_save"):
        return cw.modules_to_save["default"]

    return cw


def freeze_old_classifier_rows(model_lora, old_classes):
    if not FREEZE_OLD_CLASSIFIER_ROWS or len(old_classes) == 0:
        print("[RankExt] No classifier rows frozen.")
        return 0

    classifier = get_trainable_classifier(model_lora)
    old_idx_cpu = torch.tensor(old_classes, dtype=torch.long)

    def weight_hook(grad):
        grad = grad.clone()
        idx = old_idx_cpu.to(grad.device)
        grad[idx, :] = 0.0
        return grad

    classifier.weight.register_hook(weight_hook)

    if classifier.bias is not None:
        def bias_hook(grad):
            grad = grad.clone()
            idx = old_idx_cpu.to(grad.device)
            grad[idx] = 0.0
            return grad

        classifier.bias.register_hook(bias_hook)

    print(f"[RankExt] frozen classifier rows count: {len(old_classes)}")

    return len(old_classes)


def compute_rankext_new_old_orth_penalty(model_lora, r_old):
    if r_old == 0:
        return torch.tensor(0.0, device=next(model_lora.parameters()).device)

    penalties = []

    for _, module in model_lora.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        if A.shape[0] <= r_old or B.shape[1] <= r_old:
            continue

        A_old = A[:r_old, :]
        B_old = B[:, :r_old]

        A_new = A[r_old:, :]
        B_new = B[:, r_old:]

        if hasattr(module, "scaling"):
            scaling = (
                module.scaling["default"]
                if isinstance(module.scaling, dict)
                else module.scaling
            )
        else:
            scaling = 1.0

        old_delta = (B_old @ A_old).detach() * float(scaling)
        new_delta = (B_new @ A_new) * float(scaling)

        old_flat = old_delta.reshape(-1)
        new_flat = new_delta.reshape(-1)

        if old_flat.norm().detach().item() < 1e-12:
            continue

        if new_flat.norm().detach().item() < 1e-12:
            continue

        denom = old_flat.norm() * new_flat.norm() + 1e-8
        cos_overlap = torch.dot(old_flat, new_flat) / denom

        penalties.append(cos_overlap.pow(2))

    if len(penalties) == 0:
        return torch.tensor(0.0, device=next(model_lora.parameters()).device)

    return torch.stack(penalties).mean()


class RankExtOrthTrainer(Trainer):
    def __init__(self, *args, r_old=0, lambda_rankext_orth=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.r_old = r_old
        self.lambda_rankext_orth = lambda_rankext_orth

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        base_loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss

        if self.r_old > 0 and self.lambda_rankext_orth > 0:
            orth_penalty = compute_rankext_new_old_orth_penalty(model, self.r_old)
            loss = base_loss + self.lambda_rankext_orth * orth_penalty
        else:
            loss = base_loss

        return (loss, outputs) if return_outputs else loss


def print_rankext_sanity(model_lora, r_old, r_total):
    trainable = [n for n, p in model_lora.named_parameters() if p.requires_grad]

    print(f"[RankExt sanity] r_old={r_old}, r_total={r_total}")
    print(f"[RankExt sanity] trainable parameter tensors: {len(trainable)}")
    print("[RankExt sanity] first trainable params:")

    for n in trainable[:30]:
        print("  ", n)

    classifier = get_trainable_classifier(model_lora)

    print("Classifier weight requires_grad:", classifier.weight.requires_grad)
    print(
        "Classifier bias requires_grad:",
        classifier.bias.requires_grad if classifier.bias is not None else None,
    )


def capture_rankext_lora_snapshot(model_lora, r_old):
    snapshot = {}

    for module_name, module in model_lora.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        A = module.lora_A["default"].weight.detach().cpu().float().clone()
        B = module.lora_B["default"].weight.detach().cpu().float().clone()

        module_name_norm = normalize_module_name(module_name)

        snapshot[module_name_norm] = {
            "A_old": A[:r_old, :].clone() if r_old > 0 else None,
            "B_old": B[:, :r_old].clone() if r_old > 0 else None,
            "A_new": A[r_old:, :].clone(),
            "B_new": B[:, r_old:].clone(),
        }

    return snapshot


def summarize_rankext_lora_change(before_snapshot, model_lora, r_old):
    old_A_changes = []
    old_B_changes = []
    new_A_changes = []
    new_B_changes = []

    for module_name, module in model_lora.named_modules():
        module_name_norm = normalize_module_name(module_name)

        if module_name_norm not in before_snapshot:
            continue

        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        A_after = module.lora_A["default"].weight.detach().cpu().float()
        B_after = module.lora_B["default"].weight.detach().cpu().float()

        before = before_snapshot[module_name_norm]

        if r_old > 0:
            old_A_changes.append(float((A_after[:r_old, :] - before["A_old"]).norm()))
            old_B_changes.append(float((B_after[:, :r_old] - before["B_old"]).norm()))

        new_A_changes.append(float((A_after[r_old:, :] - before["A_new"]).norm()))
        new_B_changes.append(float((B_after[:, r_old:] - before["B_new"]).norm()))

    def safe_mean(xs):
        return float(np.mean(xs)) if len(xs) > 0 else np.nan

    def safe_max(xs):
        return float(np.max(xs)) if len(xs) > 0 else np.nan

    return {
        "old_A_change_mean": safe_mean(old_A_changes),
        "old_B_change_mean": safe_mean(old_B_changes),
        "new_A_change_mean": safe_mean(new_A_changes),
        "new_B_change_mean": safe_mean(new_B_changes),
        "old_A_change_max": safe_max(old_A_changes),
        "old_B_change_max": safe_max(old_B_changes),
        "new_A_change_max": safe_max(new_A_changes),
        "new_B_change_max": safe_max(new_B_changes),
    }


def capture_classifier_snapshot(model_lora):
    classifier = get_trainable_classifier(model_lora)

    return {
        "weight": classifier.weight.detach().cpu().clone(),
        "bias": (
            classifier.bias.detach().cpu().clone()
            if classifier.bias is not None
            else None
        ),
    }


def summarize_classifier_change(before_snapshot, model_lora, old_classes, current_classes):
    classifier = get_trainable_classifier(model_lora)

    after_w = classifier.weight.detach().cpu()
    before_w = before_snapshot["weight"]

    delta = after_w - before_w
    row_norm = delta.norm(dim=1)

    old_classes = list(old_classes)
    current_classes = list(current_classes)

    def safe_mean(indexes):
        if len(indexes) == 0:
            return np.nan
        return float(row_norm[indexes].mean().item())

    def safe_max(indexes):
        if len(indexes) == 0:
            return np.nan
        return float(row_norm[indexes].max().item())

    return {
        "classifier_old_rows_change_mean": safe_mean(old_classes),
        "classifier_current_rows_change_mean": safe_mean(current_classes),
        "classifier_old_rows_change_max": safe_max(old_classes),
        "classifier_current_rows_change_max": safe_max(current_classes),
    }


def save_classifier_snapshot(model_lora, step_idx):
    classifier = get_trainable_classifier(model_lora)

    rankext_classifier_snapshots[step_idx] = {
        "weight": classifier.weight.detach().cpu().clone(),
        "bias": (
            classifier.bias.detach().cpu().clone()
            if classifier.bias is not None
            else None
        ),
    }

    print(f"[RankExt] saved classifier snapshot for step {step_idx + 1}")


def apply_stitched_classifier_to_rankext(model_lora):
    classifier = get_trainable_classifier(model_lora)

    with torch.no_grad():
        for step_idx in range(num_steps):
            if step_idx not in rankext_classifier_snapshots:
                continue

            class_ids = classes_for_step(step_idx)
            snap = rankext_classifier_snapshots[step_idx]

            w = snap["weight"].to(
                device=classifier.weight.device,
                dtype=classifier.weight.dtype,
            )

            classifier.weight[class_ids, :] = w[class_ids, :]

            if classifier.bias is not None and snap["bias"] is not None:
                b = snap["bias"].to(
                    device=classifier.bias.device,
                    dtype=classifier.bias.dtype,
                )
                classifier.bias[class_ids] = b[class_ids]

    print("[RankExt + Stitched] classifier rows applied.")


def rankext_prediction_group_stats(trainer, eval_dataset, eval_name, current_classes, first_classes, later_classes, seen_classes, step_idx):
    pred_output = trainer.predict(eval_dataset)
    logits = pred_output.predictions
    labels = pred_output.label_ids
    preds = np.argmax(logits, axis=1)

    current_classes = set(current_classes)
    first_classes = set(first_classes)
    later_classes = set(later_classes)
    seen_classes = set(seen_classes)

    return {
        "step": step_idx + 1,
        "eval_name": eval_name,
        "accuracy_from_predict": float((preds == labels).mean()),
        "pred_frac_current_classes": float(np.mean([p in current_classes for p in preds])),
        "pred_frac_first_step_classes": float(np.mean([p in first_classes for p in preds])),
        "pred_frac_later_seen_classes": float(np.mean([p in later_classes for p in preds])),
        "pred_frac_seen_classes": float(np.mean([p in seen_classes for p in preds])),
        "pred_frac_unseen_classes": float(np.mean([p not in seen_classes for p in preds])),
        "label_min": int(np.min(labels)),
        "label_max": int(np.max(labels)),
        "pred_min": int(np.min(preds)),
        "pred_max": int(np.max(preds)),
        "num_examples": int(len(labels)),
    }


prev_rankext_adapter_dir = None
last_rankext_model = None

for step_idx in range(num_steps):
    cifar_step = step_idx + 1

    r_old = step_idx * RANKEXT_NEW_RANK_PER_STEP
    r_total = (step_idx + 1) * RANKEXT_NEW_RANK_PER_STEP
    rankext_alpha_current = RANKEXT_ALPHA_MULTIPLIER * r_total

    current_classes = classes_for_step(step_idx)

    old_classes = []
    for s in range(step_idx):
        old_classes.extend(classes_for_step(s))

    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    print("\n" + "=" * 80)
    print(f"[RankExt] Step {cifar_step}")
    print("Current classes:", current_classes[:5], "...", current_classes[-5:])
    print("Old classes count:", len(old_classes))
    print(f"Rank: old={r_old}, total={r_total}, alpha={rankext_alpha_current}")
    print("Train size:", len(train_step), "| Test size:", len(test_step))
    print(f"lambda_rankext_orth = {LAMBDA_RANKEXT_ORTH}")
    print(f"rankext_weight_decay = {RANKEXT_WEIGHT_DECAY}")
    print(f"use_rankext_ipc_lite = {USE_RANKEXT_IPC_LITE}")
    print("=" * 80)

    base_model = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)

    rankext_config = LoraConfig(
        r=r_total,
        lora_alpha=rankext_alpha_current,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=TARGET_MODULES,
        modules_to_save=["classifier"],
    )

    model_rankext = get_peft_model(base_model, rankext_config)

    copy_info = copy_previous_rankext_weights(
        model_lora=model_rankext,
        prev_adapter_dir=prev_rankext_adapter_dir,
        r_old=r_old,
    )

    freeze_info = freeze_old_lora_ranks(model_rankext, r_old)
    frozen_classifier_rows = freeze_old_classifier_rows(model_rankext, old_classes)

    print_rankext_sanity(model_rankext, r_old, r_total)

    before_lora_snapshot = capture_rankext_lora_snapshot(model_rankext, r_old)
    before_classifier_snapshot = capture_classifier_snapshot(model_rankext)

    step_train_dir = os.path.join(
        "outputs",
        RUN_NAME,
        f"step_{cifar_step}_rankext_r{r_total}",
    )

    args_rankext = TrainingArguments(
        output_dir=step_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=RANKEXT_EPOCHS,
        learning_rate=LR_RANKEXT,
        weight_decay=RANKEXT_WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_LORA,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_LORA,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_rankext = RankExtOrthTrainer(
        model=model_rankext,
        args=args_rankext,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        r_old=r_old,
        lambda_rankext_orth=LAMBDA_RANKEXT_ORTH,
    )

    print("\n[RankExt] Before training current-step evaluation")
    before_current = trainer_rankext.evaluate(eval_dataset=test_step)
    print(before_current)

    trainer_rankext.train()

    eval_current = trainer_rankext.evaluate(eval_dataset=test_step)

    eval_first = make_eval_dataset(classes_for_step(0))
    seen_now = classes_for_cumulative(step_idx)
    later_now = [c for c in seen_now if c not in classes_for_step(0)]

    eval_later = make_eval_dataset(later_now) if len(later_now) > 0 else None
    eval_seen = make_eval_dataset(seen_now)

    eval_first_res = trainer_rankext.evaluate(eval_dataset=eval_first)
    eval_later_res = (
        trainer_rankext.evaluate(eval_dataset=eval_later)
        if eval_later is not None
        else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    )
    eval_seen_res = trainer_rankext.evaluate(eval_dataset=eval_seen)

    lora_change_info = summarize_rankext_lora_change(
        before_snapshot=before_lora_snapshot,
        model_lora=trainer_rankext.model,
        r_old=r_old,
    )

    classifier_change_info = summarize_classifier_change(
        before_snapshot=before_classifier_snapshot,
        model_lora=trainer_rankext.model,
        old_classes=old_classes,
        current_classes=current_classes,
    )

    pred_debug_current = rankext_prediction_group_stats(
        trainer=trainer_rankext,
        eval_dataset=test_step,
        eval_name="current_step",
        current_classes=current_classes,
        first_classes=classes_for_step(0),
        later_classes=later_now,
        seen_classes=seen_now,
        step_idx=step_idx,
    )

    rankext_debug_row = {
        "step": cifar_step,
        "r_old": r_old,
        "r_total": r_total,
        "rank_alpha": rankext_alpha_current,
        "before_current_accuracy": float(before_current.get("eval_accuracy", np.nan)),
        "after_current_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
        "first_step_accuracy": float(eval_first_res.get("eval_accuracy", np.nan)),
        "later_steps_accuracy": float(eval_later_res.get("eval_accuracy", np.nan)),
        "all_seen_accuracy": float(eval_seen_res.get("eval_accuracy", np.nan)),
        "copied_A": copy_info["copied_A"],
        "copied_B": copy_info["copied_B"],
        "copied_classifier": copy_info["copied_classifier"],
        "frozen_A_modules": freeze_info["frozen_A_modules"],
        "frozen_B_modules": freeze_info["frozen_B_modules"],
        "frozen_classifier_rows": frozen_classifier_rows,
        **lora_change_info,
        **classifier_change_info,
        **{
            f"pred_current_{k}": v
            for k, v in pred_debug_current.items()
            if k not in ["step", "eval_name"]
        },
    }

    rankext_debug_rows.append(rankext_debug_row)

    pd.DataFrame(trainer_rankext.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{cifar_step}_rankext_log_history.csv"),
        index=False,
    )

    rankext_results.append({
        "experiment": f"rankext_step_{cifar_step}",
        "method": "rank_extension",
        "step": cifar_step,
        "rank_total": r_total,
        "rank_alpha": rankext_alpha_current,
        "eval_type": "current_step",
        "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
        "eval_loss": float(eval_current.get("eval_loss", np.nan)),
    })
    rankext_results.append({
        "experiment": f"rankext_step_{cifar_step}",
        "method": "rank_extension",
        "step": cifar_step,
        "rank_total": r_total,
        "rank_alpha": rankext_alpha_current,
        "eval_type": "first_step",
        "eval_accuracy": float(eval_first_res.get("eval_accuracy", np.nan)),
        "eval_loss": float(eval_first_res.get("eval_loss", np.nan)),
    })
    rankext_results.append({
        "experiment": f"rankext_step_{cifar_step}",
        "method": "rank_extension",
        "step": cifar_step,
        "rank_total": r_total,
        "rank_alpha": rankext_alpha_current,
        "eval_type": "later_steps",
        "eval_accuracy": float(eval_later_res.get("eval_accuracy", np.nan)),
        "eval_loss": float(eval_later_res.get("eval_loss", np.nan)),
    })
    rankext_results.append({
        "experiment": f"rankext_step_{cifar_step}",
        "method": "rank_extension",
        "step": cifar_step,
        "rank_total": r_total,
        "rank_alpha": rankext_alpha_current,
        "eval_type": "all_seen",
        "eval_accuracy": float(eval_seen_res.get("eval_accuracy", np.nan)),
        "eval_loss": float(eval_seen_res.get("eval_loss", np.nan)),
    })

    save_classifier_snapshot(trainer_rankext.model, step_idx)

    rankext_save_dir = os.path.join(
        RANKEXT_BANK_DIR,
        f"step_{cifar_step}_rankext_rank_{r_total}_adapter",
    )

    trainer_rankext.model.save_pretrained(rankext_save_dir)
    image_processor.save_pretrained(rankext_save_dir)

    prev_rankext_adapter_dir = rankext_save_dir
    last_rankext_model = trainer_rankext.model

    print(f"[RankExt] Step {cifar_step} current:", eval_current)
    print(f"[RankExt] Step {cifar_step} first  :", eval_first_res)
    print(f"[RankExt] Step {cifar_step} later  :", eval_later_res)
    print(f"[RankExt] Step {cifar_step} seen   :", eval_seen_res)

    print("[RankExt debug row]")
    print(pd.DataFrame([rankext_debug_row]).T)


rankext_debug_df = pd.DataFrame(rankext_debug_rows)
rankext_debug_path = os.path.join(TABLES_DIR, "rankext_debug_diagnostics.csv")
rankext_debug_df.to_csv(rankext_debug_path, index=False)

print("\n" + "=" * 80)
print("[RankExt] Debug diagnostics saved:")
print(rankext_debug_path)
print(rankext_debug_df)
print("=" * 80)


print("\n" + "=" * 80)
print("[RankExt + Stitched] Final evaluation")
print("=" * 80)

apply_stitched_classifier_to_rankext(last_rankext_model)

final_first_classes = classes_for_step(0)

final_later_classes = []
for s in range(1, num_steps):
    final_later_classes.extend(classes_for_step(s))

final_seen_classes = classes_for_cumulative(num_steps - 1)

eval_first_final = make_eval_dataset(
    final_first_classes,
    name="rankext_stitched_first",
)
eval_later_final = make_eval_dataset(
    final_later_classes,
    name="rankext_stitched_later",
)
eval_seen_final = make_eval_dataset(
    final_seen_classes,
    name="rankext_stitched_seen",
)

trainer_rankext_stitched = Trainer(
    model=last_rankext_model,
    args=TrainingArguments(
        output_dir=os.path.join(
            "outputs",
            RUN_NAME,
            "eval_rankext_stitched_final",
        ),
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

stitched_first = trainer_rankext_stitched.evaluate(eval_dataset=eval_first_final)
stitched_later = trainer_rankext_stitched.evaluate(eval_dataset=eval_later_final)
stitched_seen = trainer_rankext_stitched.evaluate(eval_dataset=eval_seen_final)

rankext_results.extend([
    {
        "experiment": f"rankext_stitched_step_{num_steps}",
        "method": "rank_extension_stitched",
        "step": num_steps,
        "rank_total": num_steps * RANKEXT_NEW_RANK_PER_STEP,
        "rank_alpha": RANKEXT_ALPHA_MULTIPLIER * num_steps * RANKEXT_NEW_RANK_PER_STEP,
        "eval_type": "first_step",
        "eval_accuracy": float(stitched_first.get("eval_accuracy", np.nan)),
        "eval_loss": float(stitched_first.get("eval_loss", np.nan)),
    },
    {
        "experiment": f"rankext_stitched_step_{num_steps}",
        "method": "rank_extension_stitched",
        "step": num_steps,
        "rank_total": num_steps * RANKEXT_NEW_RANK_PER_STEP,
        "rank_alpha": RANKEXT_ALPHA_MULTIPLIER * num_steps * RANKEXT_NEW_RANK_PER_STEP,
        "eval_type": "later_steps",
        "eval_accuracy": float(stitched_later.get("eval_accuracy", np.nan)),
        "eval_loss": float(stitched_later.get("eval_loss", np.nan)),
    },
    {
        "experiment": f"rankext_stitched_step_{num_steps}",
        "method": "rank_extension_stitched",
        "step": num_steps,
        "rank_total": num_steps * RANKEXT_NEW_RANK_PER_STEP,
        "rank_alpha": RANKEXT_ALPHA_MULTIPLIER * num_steps * RANKEXT_NEW_RANK_PER_STEP,
        "eval_type": "all_seen",
        "eval_accuracy": float(stitched_seen.get("eval_accuracy", np.nan)),
        "eval_loss": float(stitched_seen.get("eval_loss", np.nan)),
    },
])

print("[RankExt + Stitched] first_step:", stitched_first)
print("[RankExt + Stitched] later_steps:", stitched_later)
print("[RankExt + Stitched] all_seen:", stitched_seen)

print("\nRankExt diagnostic finished.")

In [ ]:
def filter_dataset_by_classes(dataset, class_ids):
    class_ids = set(class_ids)
    return dataset.filter(lambda x: x["label"] in class_ids)

## merging method

In [ ]:
print("\n===== MERGE EVALUATION: Simple AVG + DO-Merge =====")

merged_final_evals = {}

variant_class_groups = [
    classes_for_step(0),
    classes_for_step(1),
    classes_for_step(2),
    classes_for_step(3),
    classes_for_step(4),
]


def evaluate_final_merged_model(merged_model, method_key, output_suffix):
    first_step_classes = classes_for_step(0)

    later_step_classes = []
    for s in range(1, num_steps):
        later_step_classes.extend(classes_for_step(s))

    final_seen_classes = classes_for_cumulative(num_steps - 1)

    eval_first = make_eval_dataset(first_step_classes, name=f"{output_suffix}_first")
    eval_later = make_eval_dataset(later_step_classes, name=f"{output_suffix}_later")
    eval_seen = make_eval_dataset(final_seen_classes, name=f"{output_suffix}_seen")

    trainer_eval = Trainer(
        model=merged_model,
        args=TrainingArguments(
            output_dir=f"outputs/{RUN_NAME}/eval_{output_suffix}_final",
            remove_unused_columns=False,
            per_device_eval_batch_size=32,
            report_to="none",
        ),
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    final_first = trainer_eval.evaluate(eval_dataset=eval_first)
    final_later = trainer_eval.evaluate(eval_dataset=eval_later)
    final_seen = trainer_eval.evaluate(eval_dataset=eval_seen)

    merged_final_evals[method_key] = {
        "first_step": final_first,
        "later_steps": final_later,
        "all_seen": final_seen,
    }

    print(f"[{method_key}] final - first_step :", final_first)
    print(f"[{method_key}] final - later_steps:", final_later)
    print(f"[{method_key}] final - all_seen   :", final_seen)


for variant_name in VARIANTS:
    print("\n" + "-" * 80)
    print(f"SIMPLE AVG MERGING VARIANT: {variant_name.upper()}")
    print("-" * 80)

    variant_adapter_dirs = [
        os.path.join(LORA_BANK_DIR, variant_name, f"step_{s}_adapter")
        for s in range(1, num_steps + 1)
    ]

    merged_model = build_fixed_avg_merged_model(
        base_model_dir=STEP1_FINAL_OUT,
        adapter_dirs=variant_adapter_dirs,
        adapter_class_groups=variant_class_groups,
    )

    evaluate_final_merged_model(
        merged_model=merged_model,
        method_key=f"simple_avg_{variant_name}",
        output_suffix=f"simple_avg_{variant_name}",
    )


if USE_DO_MERGE:
    print("\n" + "-" * 80)
    print(f"DO-MERGE using source variant: {DO_MERGE_SOURCE_VARIANT.upper()}")
    print("-" * 80)

    do_adapter_dirs = [
        os.path.join(LORA_BANK_DIR, DO_MERGE_SOURCE_VARIANT, f"step_{s}_adapter")
        for s in range(1, num_steps + 1)
    ]

    do_merged_model = build_do_merged_model(
        base_model_dir=STEP1_FINAL_OUT,
        adapter_dirs=do_adapter_dirs,
        adapter_class_groups=variant_class_groups,
    )

    evaluate_final_merged_model(
        merged_model=do_merged_model,
        method_key=f"do_merge_{DO_MERGE_SOURCE_VARIANT}",
        output_suffix=f"do_merge_{DO_MERGE_SOURCE_VARIANT}",
    )

## 9) Baseline: full fine-tune instead of LoRA (Step 2)

In [ ]:
ft_results = []

for s in range(1, num_steps + 1):
    stale_train_dir = f"outputs/{RUN_NAME}/step_{s}_ft"
    stale_final_dir = f"outputs/{RUN_NAME}/step_{s}_ft_final"
    if os.path.exists(stale_train_dir):
        shutil.rmtree(stale_train_dir)
    if os.path.exists(stale_final_dir):
        shutil.rmtree(stale_final_dir)

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

def _clone_classifier_state(model):
    out = {}
    out["weight"] = model.classifier.weight.detach().cpu().clone()
    if model.classifier.bias is not None:
        out["bias"] = model.classifier.bias.detach().cpu().clone()
    else:
        out["bias"] = None
    return out

def _classifier_debug_stats(before_state, after_model, current_classes, seen_classes, step_idx):
    after_w = after_model.classifier.weight.detach().cpu()
    before_w = before_state["weight"]

    delta_w = after_w - before_w
    row_delta_norm = delta_w.norm(dim=1)
    row_after_norm = after_w.norm(dim=1)

    current_classes = list(current_classes)
    seen_classes = list(seen_classes)
    old_seen_classes = [c for c in seen_classes if c not in current_classes]
    unseen_classes = [c for c in range(num_classes) if c not in seen_classes]

    def safe_mean(indexes, values):
        if len(indexes) == 0:
            return np.nan
        return float(values[indexes].mean().item())

    def safe_max(indexes, values):
        if len(indexes) == 0:
            return np.nan
        return float(values[indexes].max().item())

    return {
        "step": step_idx + 1,
        "current_classes_min": int(min(current_classes)),
        "current_classes_max": int(max(current_classes)),
        "seen_classes_count": int(len(seen_classes)),
        "old_seen_classes_count": int(len(old_seen_classes)),
        "unseen_classes_count": int(len(unseen_classes)),

        "classifier_delta_current_mean": safe_mean(current_classes, row_delta_norm),
        "classifier_delta_old_seen_mean": safe_mean(old_seen_classes, row_delta_norm),
        "classifier_delta_unseen_mean": safe_mean(unseen_classes, row_delta_norm),

        "classifier_delta_current_max": safe_max(current_classes, row_delta_norm),
        "classifier_delta_old_seen_max": safe_max(old_seen_classes, row_delta_norm),
        "classifier_delta_unseen_max": safe_max(unseen_classes, row_delta_norm),

        "classifier_norm_current_mean": safe_mean(current_classes, row_after_norm),
        "classifier_norm_old_seen_mean": safe_mean(old_seen_classes, row_after_norm),
        "classifier_norm_unseen_mean": safe_mean(unseen_classes, row_after_norm),
    }

def _prediction_group_stats(trainer, eval_dataset, eval_name, current_classes, first_classes, later_classes, seen_classes, step_idx):
    pred_output = trainer.predict(eval_dataset)
    logits = pred_output.predictions
    labels = pred_output.label_ids

    preds = np.argmax(logits, axis=1)

    current_classes = set(current_classes)
    first_classes = set(first_classes)
    later_classes = set(later_classes)
    seen_classes = set(seen_classes)

    pred_current = np.array([p in current_classes for p in preds], dtype=np.float32).mean()
    pred_first = np.array([p in first_classes for p in preds], dtype=np.float32).mean()
    pred_later = np.array([p in later_classes for p in preds], dtype=np.float32).mean()
    pred_seen = np.array([p in seen_classes for p in preds], dtype=np.float32).mean()
    pred_unseen = np.array([p not in seen_classes for p in preds], dtype=np.float32).mean()

    acc = float((preds == labels).mean())

    return {
        "step": step_idx + 1,
        "eval_name": eval_name,
        "accuracy_from_predict": acc,
        "pred_frac_current_classes": float(pred_current),
        "pred_frac_first_step_classes": float(pred_first),
        "pred_frac_later_seen_classes": float(pred_later),
        "pred_frac_seen_classes": float(pred_seen),
        "pred_frac_unseen_classes": float(pred_unseen),
        "num_examples": int(len(labels)),
    }

ft_debug_rows = []
ft_pred_debug_rows = []

base_model_dir = STEP1_FINAL_OUT

for step_idx in range(num_steps):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    print("\n" + "=" * 80)
    print(f"[FT DEBUG] Step {step_idx+1}")
    print("=" * 80)
    print("Loading model from:", base_model_dir)
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    tr_min, tr_max = _label_range(train_step)
    te_min, te_max = _label_range(test_step)

    print("Train label range:", tr_min, tr_max)
    print("Test label range:", te_min, te_max)

    ft_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    assert ft_model.config.num_labels == num_classes
    assert ft_model.classifier.out_features == num_classes

    before_classifier_state = _clone_classifier_state(ft_model)

    step_ft_train_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft"

    args_ft = TrainingArguments(
        output_dir=step_ft_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        num_train_epochs=FT_EPOCHS,
        learning_rate=LR_FT,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft = Trainer(
        model=ft_model,
        args=args_ft,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    seen_now = classes_for_cumulative(step_idx)
    later_now = [c for c in seen_now if c not in classes_for_step(0)]

    eval_first_ds = make_eval_dataset(classes_for_step(0), name=f"ft_step{step_idx+1}_first_step")
    eval_later_ds = make_eval_dataset(later_now, name=f"ft_step{step_idx+1}_later_steps") if len(later_now) > 0 else None
    eval_seen_ds = make_eval_dataset(seen_now, name=f"ft_step{step_idx+1}_all_seen")

    print("\n[FT DEBUG] Before training evaluation")
    before_current = trainer_ft.evaluate(eval_dataset=test_step)
    before_first = trainer_ft.evaluate(eval_dataset=eval_first_ds)
    before_later = (
        trainer_ft.evaluate(eval_dataset=eval_later_ds)
        if eval_later_ds is not None
        else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    )
    before_seen = trainer_ft.evaluate(eval_dataset=eval_seen_ds)

    print("Before current_step:", before_current)
    print("Before first_step:", before_first)
    print("Before later_steps:", before_later)
    print("Before all_seen:", before_seen)

    trainer_ft.train()

    print("\n[FT DEBUG] After training evaluation")
    eval_current = trainer_ft.evaluate(eval_dataset=test_step)
    eval_first = trainer_ft.evaluate(eval_dataset=eval_first_ds)
    eval_later = (
        trainer_ft.evaluate(eval_dataset=eval_later_ds)
        if eval_later_ds is not None
        else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    )
    eval_seen = trainer_ft.evaluate(eval_dataset=eval_seen_ds)

    print("After current_step:", eval_current)
    print("After first_step:", eval_first)
    print("After later_steps:", eval_later)
    print("After all_seen:", eval_seen)

    pd.DataFrame(trainer_ft.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_log_history.csv"),
        index=False
    )

    step_ft_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_final"
    os.makedirs(step_ft_dir, exist_ok=True)

    trainer_ft.save_model(step_ft_dir)
    image_processor.save_pretrained(step_ft_dir)

    ft_debug_rows.append(
        _classifier_debug_stats(
            before_state=before_classifier_state,
            after_model=trainer_ft.model,
            current_classes=class_ids,
            seen_classes=seen_now,
            step_idx=step_idx,
        )
    )

    ft_pred_debug_rows.append(
        _prediction_group_stats(
            trainer=trainer_ft,
            eval_dataset=test_step,
            eval_name="current_step",
            current_classes=class_ids,
            first_classes=classes_for_step(0),
            later_classes=later_now,
            seen_classes=seen_now,
            step_idx=step_idx,
        )
    )

    ft_pred_debug_rows.append(
        _prediction_group_stats(
            trainer=trainer_ft,
            eval_dataset=eval_first_ds,
            eval_name="first_step",
            current_classes=class_ids,
            first_classes=classes_for_step(0),
            later_classes=later_now,
            seen_classes=seen_now,
            step_idx=step_idx,
        )
    )

    if eval_later_ds is not None:
        ft_pred_debug_rows.append(
            _prediction_group_stats(
                trainer=trainer_ft,
                eval_dataset=eval_later_ds,
                eval_name="later_steps",
                current_classes=class_ids,
                first_classes=classes_for_step(0),
                later_classes=later_now,
                seen_classes=seen_now,
                step_idx=step_idx,
            )
        )

    ft_pred_debug_rows.append(
        _prediction_group_stats(
            trainer=trainer_ft,
            eval_dataset=eval_seen_ds,
            eval_name="all_seen",
            current_classes=class_ids,
            first_classes=classes_for_step(0),
            later_classes=later_now,
            seen_classes=seen_now,
            step_idx=step_idx,
        )
    )

    ft_results.extend([
        {
            "experiment": "ft_step_eval",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
        },
        {
            "experiment": "ft_step_eval",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(eval_first.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_first.get("eval_loss", np.nan)),
        },
        {
            "experiment": "ft_step_eval",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "later_steps",
            "eval_accuracy": float(eval_later.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_later.get("eval_loss", np.nan)),
        },
        {
            "experiment": "ft_step_eval",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(eval_seen.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_seen.get("eval_loss", np.nan)),
        },
    ])

    base_model_dir = step_ft_dir

ft_debug_df = pd.DataFrame(ft_debug_rows)
ft_pred_debug_df = pd.DataFrame(ft_pred_debug_rows)
ft_results_debug_df = pd.DataFrame(ft_results)

ft_debug_df.to_csv(os.path.join(TABLES_DIR, "ft_debug_classifier_drift.csv"), index=False)
ft_pred_debug_df.to_csv(os.path.join(TABLES_DIR, "ft_debug_prediction_groups.csv"), index=False)
ft_results_debug_df.to_csv(os.path.join(TABLES_DIR, "ft_results_debug.csv"), index=False)

print("\n" + "=" * 80)
print("FT DEBUG SUMMARY: classifier drift")
print("=" * 80)
print(ft_debug_df)

print("\n" + "=" * 80)
print("FT DEBUG SUMMARY: prediction groups")
print("=" * 80)
print(ft_pred_debug_df)

print("\nSaved:")
print(os.path.join(TABLES_DIR, "ft_debug_classifier_drift.csv"))
print(os.path.join(TABLES_DIR, "ft_debug_prediction_groups.csv"))
print(os.path.join(TABLES_DIR, "ft_results_debug.csv"))

In [ ]:
ft_replay_results = []

for s in range(1, num_steps + 1):
    stale_train_dir = f"outputs/{RUN_NAME}/step_{s}_ft_replay"
    stale_final_dir = f"outputs/{RUN_NAME}/step_{s}_ft_replay_final"
    if os.path.exists(stale_train_dir):
        shutil.rmtree(stale_train_dir)
    if os.path.exists(stale_final_dir):
        shutil.rmtree(stale_final_dir)

first_step_classes = classes_for_step(0)

def build_fullft_replay_train_dataset(step_idx, replay_per_old_step=250):
    current_classes = classes_for_step(step_idx)

    current_train_plain = filter_by_classes(dataset["train"], current_classes)
    replay_parts = []

    if step_idx > 0:
        for old_step_idx in range(step_idx):
            old_classes = classes_for_step(old_step_idx)
            old_train_plain = filter_by_classes(dataset["train"], old_classes)

            n_take = min(replay_per_old_step, len(old_train_plain))
            old_sample = old_train_plain.shuffle(seed=42 + step_idx * 100 + old_step_idx).select(range(n_take))

            replay_parts.append(old_sample)

            print(
                f"[FT+Replay] Step {step_idx+1}: replay from old step {old_step_idx+1} "
                f"| classes {old_classes[0]}-{old_classes[-1]} | samples={n_take}"
            )

    if len(replay_parts) > 0:
        train_plain = concatenate_datasets([current_train_plain] + replay_parts)
        train_plain = train_plain.shuffle(seed=1234 + step_idx)
    else:
        train_plain = current_train_plain.shuffle(seed=1234 + step_idx)

    train_step = train_plain.with_transform(preprocess_train)

    test_step = filter_by_classes(dataset["test"], current_classes)
    test_step = test_step.with_transform(preprocess_val)

    return train_step, test_step, current_classes


def _label_range_replay(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)


ft_replay_base_model_dir = STEP1_FINAL_OUT

for step_idx in range(num_steps):
    train_step, test_step, class_ids = build_fullft_replay_train_dataset(
        step_idx=step_idx,
        replay_per_old_step=FT_REPLAY_PER_OLD_STEP,
    )

    print("\n" + "=" * 80)
    print(f"[FT+Replay] Step {step_idx+1}")
    print("=" * 80)
    print("Loading model from:", ft_replay_base_model_dir)
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])
    print("Train size with replay:", len(train_step))
    print("Test size current step:", len(test_step))

    tr_min, tr_max = _label_range_replay(train_step)
    te_min, te_max = _label_range_replay(test_step)

    print("Train label range:", tr_min, tr_max)
    print("Test label range:", te_min, te_max)

    ft_replay_model = AutoModelForImageClassification.from_pretrained(ft_replay_base_model_dir)

    assert ft_replay_model.config.num_labels == num_classes
    assert ft_replay_model.classifier.out_features == num_classes

    step_ft_replay_train_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_replay"

    args_ft_replay = TrainingArguments(
        output_dir=step_ft_replay_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        num_train_epochs=FT_REPLAY_EPOCHS,
        learning_rate=LR_FT_REPLAY,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft_replay = Trainer(
        model=ft_replay_model,
        args=args_ft_replay,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_ft_replay.train()

    eval_current = trainer_ft_replay.evaluate(eval_dataset=test_step)

    pd.DataFrame(trainer_ft_replay.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_replay_log_history.csv"),
        index=False
    )

    step_ft_replay_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_replay_final"
    os.makedirs(step_ft_replay_dir, exist_ok=True)

    trainer_ft_replay.save_model(step_ft_replay_dir)
    image_processor.save_pretrained(step_ft_replay_dir)

    seen_now = classes_for_cumulative(step_idx)
    later_now = [c for c in seen_now if c not in classes_for_step(0)]

    eval_first = trainer_ft_replay.evaluate(
        eval_dataset=make_eval_dataset(classes_for_step(0), name=f"ft_replay_step{step_idx+1}_first_step")
    )

    eval_later = (
        trainer_ft_replay.evaluate(
            eval_dataset=make_eval_dataset(later_now, name=f"ft_replay_step{step_idx+1}_later_steps")
        )
        if len(later_now) > 0
        else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    )

    eval_seen = trainer_ft_replay.evaluate(
        eval_dataset=make_eval_dataset(seen_now, name=f"ft_replay_step{step_idx+1}_all_seen")
    )

    ft_replay_results.extend([
        {
            "experiment": "ft_replay_step_eval",
            "method": "full_finetune_replay",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
            "replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
        },
        {
            "experiment": "ft_replay_step_eval",
            "method": "full_finetune_replay",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(eval_first.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_first.get("eval_loss", np.nan)),
            "replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
        },
        {
            "experiment": "ft_replay_step_eval",
            "method": "full_finetune_replay",
            "step": step_idx + 1,
            "eval_type": "later_steps",
            "eval_accuracy": float(eval_later.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_later.get("eval_loss", np.nan)),
            "replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
        },
        {
            "experiment": "ft_replay_step_eval",
            "method": "full_finetune_replay",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(eval_seen.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_seen.get("eval_loss", np.nan)),
            "replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
        },
    ])

    ft_replay_base_model_dir = step_ft_replay_dir


final_ft_replay_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_replay_final"
final_ft_replay_model = AutoModelForImageClassification.from_pretrained(final_ft_replay_dir)

args_ft_replay_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_replay_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_replay_final = Trainer(
    model=final_ft_replay_model,
    args=args_ft_replay_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

ft_replay_test_first = make_eval_dataset(first_step_classes)
ft_replay_test_later = make_eval_dataset(later_step_classes)
ft_replay_test_seen = make_eval_dataset(final_seen_classes)

print("Final FT+Replay num_labels:", final_ft_replay_model.config.num_labels)
print("Final FT+Replay classifier out_features:", final_ft_replay_model.classifier.out_features)
assert final_ft_replay_model.classifier.out_features == num_classes

ft_replay_final_first = trainer_ft_replay_final.evaluate(ft_replay_test_first)
ft_replay_final_later = trainer_ft_replay_final.evaluate(ft_replay_test_later)
ft_replay_final_seen = trainer_ft_replay_final.evaluate(ft_replay_test_seen)

print("FT+Replay final - first step:", ft_replay_final_first)
print("FT+Replay final - later steps:", ft_replay_final_later)
print("FT+Replay final - all seen:", ft_replay_final_seen)

ft_replay_results_df = pd.DataFrame(ft_replay_results)
ft_replay_results_df.to_csv(
    os.path.join(TABLES_DIR, "ft_replay_results_debug.csv"),
    index=False,
)

print("\nSaved:")
print(os.path.join(TABLES_DIR, "ft_replay_results_debug.csv"))
print(ft_replay_results_df)

In [ ]:
final_ft_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_final"
final_ft_model = AutoModelForImageClassification.from_pretrained(final_ft_dir)

args_ft_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_final = Trainer(
    model=final_ft_model,
    args=args_ft_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

ft_test_first = make_eval_dataset(first_step_classes)
ft_test_later = make_eval_dataset(later_step_classes)
ft_test_seen = make_eval_dataset(final_seen_classes)

print("Final FT num_labels:", final_ft_model.config.num_labels)
print("Final FT classifier out_features:", final_ft_model.classifier.out_features)
assert final_ft_model.classifier.out_features == num_classes

ft_final_first = trainer_ft_final.evaluate(ft_test_first)
ft_final_later = trainer_ft_final.evaluate(ft_test_later)
ft_final_seen = trainer_ft_final.evaluate(ft_test_seen)

print("FT final - first step:", ft_final_first)
print("FT final - later steps:", ft_final_later)
print("FT final - all seen:", ft_final_seen)

def final_ft_prediction_collapse_check(trainer, eval_dataset, eval_name):
    pred_output = trainer.predict(eval_dataset)
    logits = pred_output.predictions
    labels = pred_output.label_ids

    preds = np.argmax(logits, axis=1)
    acc = float((preds == labels).mean())

    first_classes_set = set(first_step_classes)
    later_classes_set = set(later_step_classes)
    final_seen_set = set(final_seen_classes)
    last_step_set = set(classes_for_step(num_steps - 1))

    row = {
        "eval_name": eval_name,
        "accuracy_from_predict": acc,
        "num_examples": int(len(labels)),
        "pred_frac_first_step": float(np.mean([p in first_classes_set for p in preds])),
        "pred_frac_later_steps": float(np.mean([p in later_classes_set for p in preds])),
        "pred_frac_last_step": float(np.mean([p in last_step_set for p in preds])),
        "pred_frac_seen": float(np.mean([p in final_seen_set for p in preds])),
        "pred_frac_unseen": float(np.mean([p not in final_seen_set for p in preds])),
        "label_min": int(np.min(labels)),
        "label_max": int(np.max(labels)),
        "pred_min": int(np.min(preds)),
        "pred_max": int(np.max(preds)),
    }

    return row

final_ft_collapse_rows = [
    final_ft_prediction_collapse_check(trainer_ft_final, ft_test_first, "first_step"),
    final_ft_prediction_collapse_check(trainer_ft_final, ft_test_later, "later_steps"),
    final_ft_prediction_collapse_check(trainer_ft_final, ft_test_seen, "all_seen"),
]

final_ft_collapse_df = pd.DataFrame(final_ft_collapse_rows)
final_ft_collapse_df.to_csv(
    os.path.join(TABLES_DIR, "final_ft_prediction_collapse_check.csv"),
    index=False
)

print("\nFinal FT prediction collapse check:")
print(final_ft_collapse_df)

print("\nSaved:")
print(os.path.join(TABLES_DIR, "final_ft_prediction_collapse_check.csv"))

## 10) Upper bound: joint training (full dataset)

In [ ]:
train_joint, test_joint, label2new_J, new2orig_J, class_ids_J = make_step_datasets(
    step_idx=0, split_type="full", remap_labels=False
)

config_joint = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

joint_model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    config=config_joint,
    ignore_mismatched_sizes=True,
)

args_joint = TrainingArguments(
    output_dir=JOINT_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=JOINT_EPOCHS,
    learning_rate=LR_JOINT,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_FT,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_FT,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer_joint = Trainer(
    model=joint_model,
    args=args_joint,
    train_dataset=train_joint,
    eval_dataset=test_joint,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer_joint.train()

eval_joint = trainer_joint.evaluate()
print({"eval_joint_full": eval_joint})

first_step_classes = classes_for_step(0)

later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(classes_for_step(s))

final_seen_classes = classes_for_cumulative(num_steps - 1)

joint_eval_first = make_eval_dataset(first_step_classes, name="joint_first")
joint_eval_later = make_eval_dataset(later_step_classes, name="joint_later")
joint_eval_seen  = make_eval_dataset(final_seen_classes, name="joint_seen")

joint_final_first = trainer_joint.evaluate(eval_dataset=joint_eval_first)
joint_final_later = trainer_joint.evaluate(eval_dataset=joint_eval_later)
joint_final_seen  = trainer_joint.evaluate(eval_dataset=joint_eval_seen)

print("Joint final - first_step:", joint_final_first)
print("Joint final - later_steps:", joint_final_later)
print("Joint final - all_seen:", joint_final_seen)

In [ ]:
joint_log_df = pd.DataFrame(trainer_joint.state.log_history)
joint_log_df.to_csv(os.path.join(TABLES_DIR, "joint_log_history.csv"), index=False)

joint_metrics = {
    "experiment": "joint_full",
    "eval_loss": float(eval_joint.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_joint.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "joint_metrics.json"), "w") as f:
    json.dump(joint_metrics, f, indent=2)

joint_log_df.tail()

In [ ]:
joint_test_first = make_eval_dataset(first_step_classes)
joint_test_later = make_eval_dataset(later_step_classes)
joint_test_all = make_eval_dataset(all_classes)

joint_final_first = trainer_joint.evaluate(joint_test_first)
joint_final_later = trainer_joint.evaluate(joint_test_later)
joint_final_all = trainer_joint.evaluate(joint_test_all)

print("Joint final - first step:", joint_final_first)
print("Joint final - later steps:", joint_final_later)
print("Joint final - all classes:", joint_final_all)

## 11) Compare results (step test vs full test)

In [ ]:
def grab_acc(d):
    return float(d["eval_accuracy"]) if "eval_accuracy" in d else np.nan

def grab_loss(d):
    return float(d["eval_loss"]) if "eval_loss" in d else np.nan

all_results = []

all_results.extend(results)
all_results.extend(lora_results)
all_results.extend(rankext_results)
all_results.extend(ft_results)

if "ft_replay_results" in globals():
    all_results.extend(ft_replay_results)

# final simple avg variants
for method_key, eval_dict in merged_final_evals.items():
    all_results.extend([
        {
            "experiment": f"{method_key}_final_eval",
            "method": method_key,
            "step": num_steps,
            "eval_type": "first_step",
            "eval_accuracy": grab_acc(eval_dict["first_step"]),
            "eval_loss": grab_loss(eval_dict["first_step"]),
        },
        {
            "experiment": f"{method_key}_final_eval",
            "method": method_key,
            "step": num_steps,
            "eval_type": "later_steps",
            "eval_accuracy": grab_acc(eval_dict["later_steps"]),
            "eval_loss": grab_loss(eval_dict["later_steps"]),
        },
        {
            "experiment": f"{method_key}_final_eval",
            "method": method_key,
            "step": num_steps,
            "eval_type": "all_seen",
            "eval_accuracy": grab_acc(eval_dict["all_seen"]),
            "eval_loss": grab_loss(eval_dict["all_seen"]),
        },
    ])

# FT without replay
all_results.extend([
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(ft_final_first),
        "eval_loss": grab_loss(ft_final_first),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(ft_final_later),
        "eval_loss": grab_loss(ft_final_later),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(ft_final_seen),
        "eval_loss": grab_loss(ft_final_seen),
    },
])

# FT with replay
if "ft_replay_final_first" in globals():
    all_results.extend([
        {
            "experiment": "ft_replay_final_eval",
            "method": "full_finetune_replay",
            "step": num_steps,
            "eval_type": "first_step",
            "eval_accuracy": grab_acc(ft_replay_final_first),
            "eval_loss": grab_loss(ft_replay_final_first),
            "replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
        },
        {
            "experiment": "ft_replay_final_eval",
            "method": "full_finetune_replay",
            "step": num_steps,
            "eval_type": "later_steps",
            "eval_accuracy": grab_acc(ft_replay_final_later),
            "eval_loss": grab_loss(ft_replay_final_later),
            "replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
        },
        {
            "experiment": "ft_replay_final_eval",
            "method": "full_finetune_replay",
            "step": num_steps,
            "eval_type": "all_seen",
            "eval_accuracy": grab_acc(ft_replay_final_seen),
            "eval_loss": grab_loss(ft_replay_final_seen),
            "replay_per_old_step": FT_REPLAY_PER_OLD_STEP,
        },
    ])

# JOINT
all_results.extend([
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(joint_final_first),
        "eval_loss": grab_loss(joint_final_first),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(joint_final_later),
        "eval_loss": grab_loss(joint_final_later),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(joint_final_seen),
        "eval_loss": grab_loss(joint_final_seen),
    },
])

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(TABLES_DIR, "all_results_raw.csv"), index=False)

print(results_df)

In [ ]:
results_df_clean = results_df.copy()

results_df_clean = results_df_clean.rename(columns={
    "eval_accuracy": "accuracy",
    "eval_loss": "loss",
    "eval_type": "eval_set",
})


def map_method(x):
    x = str(x)

    if x == "lora_normal":
        return "LoRA Normal"

    if x == "lora_orth":
        return "LoRA Orth"

    if x == "lora_replay":
        return "LoRA Replay"

    if x == "rank_extension":
        return "RankExt"

    if x == "rank_extension_stitched":
        return "RankExt + Stitched"

    if x == "rank_extension_ipc_lite":
        return "RankExt-IPC-lite"

    if x == "rank_extension_ipc_lite_stitched":
        return "RankExt-IPC-lite + Stitched"

    if x == "simple_avg_normal":
        return "Simple AVG - Normal"

    if x == "simple_avg_orth":
        return "Simple AVG - Orth"

    if x == "simple_avg_replay":
        return "Simple AVG - Replay"

    if x == "do_merge_normal":
        return "DO-Merge - Normal"

    if x == "do_merge_orth":
        return "DO-Merge - Orth"

    if x == "do_merge_replay":
        return "DO-Merge - Replay"

    if x == "full_finetune":
        return "Full FT"

    if x == "full_finetune_replay":
        return "Full FT + Replay"

    if x == "joint_upper_bound":
        return "Joint"

    return x


results_df_clean["method"] = results_df_clean["method"].apply(map_method)


def map_eval_stage(exp_name):
    exp_name = str(exp_name)

    if "final_eval" in exp_name:
        return "final"

    if exp_name.startswith("rankext_ipc_lite_stitched_step"):
        return "final"

    if exp_name.startswith("rankext_stitched_step"):
        return "final"

    if exp_name.startswith("rankext_step"):
        return "final" if exp_name.endswith(str(num_steps)) else "step"

    if exp_name.startswith("do_merge_"):
        return "final"

    if exp_name.startswith("simple_avg_"):
        return "final"

    if exp_name == "step1_scratch":
        return "step1"

    return "step"


results_df_clean["eval_stage"] = results_df_clean["experiment"].apply(map_eval_stage)


def map_adapter_name(row):
    method = row["method"]

    if method in ["LoRA Normal", "LoRA Orth", "LoRA Replay"]:
        step = row.get("step", np.nan)
        return f"L{int(step)}" if pd.notna(step) else np.nan

    if method == "RankExt":
        r = row.get("rank_total", np.nan)
        return f"RankExt-r{int(r)}" if pd.notna(r) else "RankExt"

    if method == "RankExt + Stitched":
        r = row.get("rank_total", np.nan)
        return f"RankExt-stitched-r{int(r)}" if pd.notna(r) else "RankExt-stitched"

    if method == "RankExt-IPC-lite":
        r = row.get("rank_total", np.nan)
        return f"RankExt-IPC-r{int(r)}" if pd.notna(r) else "RankExt-IPC"

    if method == "RankExt-IPC-lite + Stitched":
        r = row.get("rank_total", np.nan)
        return f"RankExt-IPC-stitched-r{int(r)}" if pd.notna(r) else "RankExt-IPC-stitched"

    if method == "Simple AVG - Normal":
        return "L12345_avg_normal"

    if method == "Simple AVG - Orth":
        return "L12345_avg_orth"

    if method == "Simple AVG - Replay":
        return "L12345_avg_replay"

    if method == "DO-Merge - Normal":
        return "DO_merge_normal"

    if method == "DO-Merge - Orth":
        return "DO_merge_orth"

    if method == "DO-Merge - Replay":
        return "DO_merge_replay"

    if method == "Full FT":
        return "full_finetune"

    if method == "Full FT + Replay":
        return "full_finetune_replay"

    if method == "Joint":
        return "joint_upper_bound"

    return np.nan


results_df_clean["adapter_name"] = results_df_clean.apply(map_adapter_name, axis=1)

keep_cols = [
    "method",
    "adapter_name",
    "step",
    "eval_stage",
    "eval_set",
    "accuracy",
    "loss",
]

if "rank_total" in results_df_clean.columns:
    keep_cols.append("rank_total")

if "rank_alpha" in results_df_clean.columns:
    keep_cols.append("rank_alpha")

if "replay_per_old_step" in results_df_clean.columns:
    keep_cols.append("replay_per_old_step")

results_df_clean = results_df_clean[keep_cols].copy()

results_df_clean.to_csv(
    os.path.join(TABLES_DIR, "results_summary_clean.csv"),
    index=False,
)

print(results_df_clean)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_final = results_df_clean[
    (results_df_clean["eval_stage"] == "final")
    & (results_df_clean["step"] == num_steps)
].copy()

df_final = df_final[
    df_final["eval_set"].isin(["first_step", "later_steps", "all_seen"])
].copy()

df_final = df_final.drop_duplicates(
    subset=["method", "eval_set"],
    keep="last",
)

print(df_final.sort_values(["eval_set", "method"]))



In [ ]:
method_order = [
    "Simple AVG - Normal",
    "Simple AVG - Orth",
    "DO-Merge - Normal",
    "Simple AVG - Replay",
    "RankExt",
    "RankExt + Stitched",
    "RankExt-IPC-lite",
    "RankExt-IPC-lite + Stitched",
    "Full FT",
    "Full FT + Replay",
    "Joint",
]

df_plot = df_final[df_final["method"].isin(method_order)].copy()

labels = ["first_step", "later_steps", "all_seen"]

pivot = df_plot.pivot_table(
    index="eval_set",
    columns="method",
    values="accuracy",
    aggfunc="mean",
).reindex(labels)

method_order_present = [m for m in method_order if m in pivot.columns]
pivot = pivot[method_order_present]

x = np.arange(len(labels))
width = min(0.09, 0.75 / max(1, len(method_order_present)))

plt.figure(figsize=(18, 5))

for i, method_name in enumerate(method_order_present):
    values = pivot[method_name].values
    positions = x + (i - (len(method_order_present) - 1) / 2) * width

    bars = plt.bar(
        positions,
        values,
        width,
        label=method_name,
    )

    for bar, val in zip(bars, values):
        if pd.notna(val):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                val + 0.01,
                f"{val:.3f}",
                ha="center",
                va="bottom",
                fontsize=7,
                rotation=90,
            )

plt.xticks(x, labels, fontsize=11)
plt.ylabel("Accuracy")
plt.ylim(0, 1.0)
plt.title("Final Accuracy Across Evaluation Splits")
plt.legend(fontsize=9, loc="upper right")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "final_accuracy_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved:", plot_path)

In [ ]:
final_wide = df_final.pivot_table(
    index="method",
    columns="eval_set",
    values="accuracy",
    aggfunc="mean",
)

final_wide = final_wide.reindex(
    [m for m in method_order if m in final_wide.index]
)

final_wide = final_wide[["first_step", "later_steps", "all_seen"]]

final_wide_percent = final_wide * 100

final_wide_percent.to_csv(
    os.path.join(TABLES_DIR, "final_accuracy_wide_percent.csv")
)

print(final_wide_percent.round(2))

In [ ]:
def get_final_acc(method_name, eval_set):
    row = df_final[
        (df_final["method"] == method_name)
        & (df_final["eval_set"] == eval_set)
    ]

    if len(row) == 0:
        return np.nan

    return float(row["accuracy"].iloc[-1])


key_methods = [
    "Simple AVG - Normal",
    "Simple AVG - Orth",
    "DO-Merge - Normal",
    "Simple AVG - Replay",
    "RankExt",
    "RankExt + Stitched",
    "RankExt-IPC-lite",
    "RankExt-IPC-lite + Stitched",
    "Full FT",
    "Full FT + Replay",
    "Joint",
]

summary_rows = []

for method_name in key_methods:
    if method_name not in df_final["method"].unique():
        continue

    first_acc = get_final_acc(method_name, "first_step")
    later_acc = get_final_acc(method_name, "later_steps")
    seen_acc = get_final_acc(method_name, "all_seen")

    joint_first = get_final_acc("Joint", "first_step")
    simple_avg_seen = get_final_acc("Simple AVG - Normal", "all_seen")

    summary_rows.append({
        "method": method_name,
        "first_step": first_acc,
        "later_steps": later_acc,
        "all_seen": seen_acc,
        "gap_to_joint_first_step": joint_first - first_acc if pd.notna(joint_first) and pd.notna(first_acc) else np.nan,
        "delta_vs_simple_avg_all_seen": seen_acc - simple_avg_seen if pd.notna(seen_acc) and pd.notna(simple_avg_seen) else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(TABLES_DIR, "final_summary_table.csv")
summary_df.to_csv(summary_path, index=False)

print(summary_df)
print("Saved:", summary_path)

In [ ]:
def make_barplot(df_sub, title, filename):
    methods = [
        "Simple AVG - Normal",
        "Simple AVG - Orth",
        "DO-Merge - Normal",
        "Simple AVG - Replay",
        "RankExt",
        "RankExt + Stitched",
        "RankExt-IPC-lite",
        "RankExt-IPC-lite + Stitched",
        "Full FT",
        "Joint",
    ]

    vals = []
    methods_present = []

    for m in methods:
        row = df_sub[df_sub["method"] == m]

        if len(row) > 0:
            methods_present.append(m)
            vals.append(float(row["accuracy"].iloc[-1]))

    plt.figure(figsize=(14, 4))
    bars = plt.bar(methods_present, vals)

    for bar, val in zip(bars, vals):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.01,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90,
        )

    plt.ylabel("Accuracy")
    plt.ylim(0, 1.0)
    plt.title(title)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved plot to:", out_path)


final_first = df_final[df_final["eval_set"] == "first_step"]
final_later = df_final[df_final["eval_set"] == "later_steps"]
final_seen = df_final[df_final["eval_set"] == "all_seen"]

make_barplot(
    final_first,
    "Final Accuracy on First-Step Classes",
    "final_first_step_all_methods.png",
)

make_barplot(
    final_later,
    "Final Accuracy on Later Classes",
    "final_later_step_all_methods.png",
)

make_barplot(
    final_seen,
    "Final Accuracy on All Seen Classes",
    "final_all_seen_all_methods.png",
)

In [ ]:
df_merge_only = df_final[df_final["method"].isin([
    "Simple AVG - Normal",
    "Simple AVG - Orth",
    "DO-Merge - Normal",
    "Simple AVG - Replay",
])].copy()

pivot_merge = df_merge_only.pivot_table(
    index="eval_set",
    columns="method",
    values="accuracy",
    aggfunc="mean",
).reindex(["first_step", "later_steps", "all_seen"])

method_order_merge = [
    "Simple AVG - Normal",
    "Simple AVG - Orth",
    "DO-Merge - Normal",
    "Simple AVG - Replay",
]

method_order_merge = [m for m in method_order_merge if m in pivot_merge.columns]
pivot_merge = pivot_merge[method_order_merge]

x = np.arange(3)
width = min(0.18, 0.75 / max(1, len(method_order_merge)))

plt.figure(figsize=(12, 5))

for i, method_name in enumerate(method_order_merge):
    positions = x + (i - (len(method_order_merge) - 1) / 2) * width
    values = pivot_merge[method_name].values

    bars = plt.bar(
        positions,
        values,
        width,
        label=method_name,
    )

    for bar, val in zip(bars, values):
        if pd.notna(val):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                val + 0.01,
                f"{val:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=90,
            )

plt.xticks(x, ["first_step", "later_steps", "all_seen"])
plt.ylabel("Accuracy")
plt.ylim(0, 1.0)
plt.title("Effect of Merge Strategy")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "merge_strategy_comparison.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)
print(pivot_merge)

In [ ]:
comparison_summary = df_final[df_final["method"].isin([
    "Simple AVG - Normal",
    "Simple AVG - Orth",
    "DO-Merge - Normal",
    "Simple AVG - Replay",
    "RankExt",
    "RankExt + Stitched",
    "RankExt-IPC-lite",
    "RankExt-IPC-lite + Stitched",
    "Full FT",
    "Full FT + Replay",
    "Joint",
])].copy()

comparison_summary = comparison_summary.sort_values(["eval_set", "method"])

comparison_summary.to_csv(
    os.path.join(TABLES_DIR, "final_method_comparison.csv"),
    index=False,
)

print(comparison_summary)

In [ ]:
normal_all = get_final_acc("Simple AVG - Normal", "all_seen")
orth_all = get_final_acc("Simple AVG - Orth", "all_seen")
do_all = get_final_acc("DO-Merge - Normal", "all_seen")
replay_all = get_final_acc("Simple AVG - Replay", "all_seen")

rankext_all = get_final_acc("RankExt", "all_seen")
rankext_stitched_all = get_final_acc("RankExt + Stitched", "all_seen")
rankext_ipc_all = get_final_acc("RankExt-IPC-lite", "all_seen")
rankext_ipc_stitched_all = get_final_acc("RankExt-IPC-lite + Stitched", "all_seen")

fullft_all = get_final_acc("Full FT", "all_seen")
fullft_replay_all = get_final_acc("Full FT + Replay", "all_seen")
joint_all = get_final_acc("Joint", "all_seen")

print("Key final all_seen accuracies:")
print("Simple AVG - Normal              :", normal_all)
print("Simple AVG - Orth                :", orth_all)
print("DO-Merge - Normal                :", do_all)
print("Simple AVG - Replay              :", replay_all)
print("RankExt                          :", rankext_all)
print("RankExt + Stitched               :", rankext_stitched_all)
print("RankExt-IPC-lite                 :", rankext_ipc_all)
print("RankExt-IPC-lite + Stitched      :", rankext_ipc_stitched_all)
print("Full FT                          :", fullft_all)
print("Full FT + Replay                 :", fullft_replay_all)
print("Joint                            :", joint_all)

print("\nInterpretation:")
print("- Full FT is the clean no-replay catastrophic-forgetting baseline.")
print("- Full FT + Replay is a replay-relaxed diagnostic baseline.")
print("- Do not replace Full FT with Full FT + Replay in the main clean comparison.")
print("- If Full FT + Replay improves strongly, then the low Full FT result is explained by forgetting, not by a broken training pipeline.")

if pd.notna(fullft_all) and pd.notna(fullft_replay_all):
    print("\nFull FT + Replay improvement over Full FT:")
    print(fullft_replay_all - fullft_all)

if pd.notna(fullft_replay_all) and pd.notna(normal_all):
    print("\nFull FT + Replay delta vs Simple AVG - Normal:")
    print(fullft_replay_all - normal_all)